# 06 — Model Evaluation

**Thesis:** Comparative Analysis of Machine Learning Algorithms for Predicting CVD Risk in a Tunisian Hospital Population

## Objective
This notebook performs the **final, decision-driving evaluation** of all four trained models on the untouched test set created in `04_ML_Preprocessing.ipynb`. No model has seen these 306 rows during training or hyperparameter tuning.

For each model we report:
- Per-class Precision, Recall, F1, and Support (LOW / INTERMEDIARY / HIGH)
- Overall Accuracy, Macro Precision, Macro Recall, Macro F1, and Weighted F1
- One-vs-Rest Macro ROC-AUC (where `predict_proba` is available)
- Confusion matrix

Model comparison is based primarily on **Macro F1**. ROC-AUC is a secondary diagnostic. **No model is retrained here.**

## Required packages
`scikit-learn`, `imbalanced-learn`, `xgboost`, `joblib`, `matplotlib`, `seaborn`

## 0. Setup

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_DIR = Path.cwd().parent
ARTIFACTS_DIR = PROJECT_DIR / "ml_artifacts"
SPLITS_DIR = ARTIFACTS_DIR / "splits"
MODELS_DIR = ARTIFACTS_DIR / "models"
FIGURES_DIR = PROJECT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 100, "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 11,
})
sns.set_style("whitegrid")

print("Artifacts dir:", ARTIFACTS_DIR)
print("Figures dir  :", FIGURES_DIR)


Artifacts dir: /home/claude/ML_HOSPITAL/ml_artifacts
Figures dir  : /home/claude/ML_HOSPITAL/figures


In [2]:
def save_fig(fig, filename):
    path = FIGURES_DIR / filename
    fig.savefig(path, bbox_inches="tight", dpi=300)
    print(f"Figure saved -> {path}")
    plt.close(fig)


## 1. Load Test Set, Configuration, and Fitted Pipelines

In [3]:
X_test = pd.read_csv(SPLITS_DIR / "X_test.csv", index_col=0)
y_test = pd.read_csv(SPLITS_DIR / "y_test.csv", index_col=0).iloc[:, 0]

with open(ARTIFACTS_DIR / "ml_config.json") as f:
    config = json.load(f)

TARGET = config["target"]
RISK_LABELS = {int(k): v for k, v in config["risk_labels"].items()}
CLASS_ORDER = [0, 1, 2]
CLASS_NAMES = [RISK_LABELS[i] for i in CLASS_ORDER]
CONTINUOUS_NUMERIC = config["continuous_numeric"]
BINARY_ORDINAL = config["binary_ordinal"]

assert X_test.shape == (306, 14), f"Unexpected X_test shape: {X_test.shape}"
assert y_test.shape == (306,),    f"Unexpected y_test shape: {y_test.shape}"

print("X_test:", X_test.shape, " y_test:", y_test.shape)
print("\nTest class distribution:")
print(y_test.value_counts().reindex(CLASS_ORDER).rename(index=RISK_LABELS))


X_test: (306, 14)  y_test: (306,)

Test class distribution:
CVD Risk Level
LOW              44
INTERMEDIARY    116
HIGH            146
Name: count, dtype: int64


In [4]:
MODEL_FILE_KEYS = {
    "Decision Tree":                   "decision_tree",
    "Random Forest":                   "random_forest",
    "Multinomial Logistic Regression": "logistic_regression",
    "XGBoost":                         "xgboost",
}

models = {}
for display_name, file_key in MODEL_FILE_KEYS.items():
    path = MODELS_DIR / f"{file_key}.joblib"
    models[display_name] = joblib.load(path)
    print(f"Loaded {display_name:35s} <- {path}")


Loaded Decision Tree                       <- /home/claude/ML_HOSPITAL/ml_artifacts/models/decision_tree.joblib
Loaded Random Forest                       <- /home/claude/ML_HOSPITAL/ml_artifacts/models/random_forest.joblib
Loaded Multinomial Logistic Regression     <- /home/claude/ML_HOSPITAL/ml_artifacts/models/logistic_regression.joblib
Loaded XGBoost                             <- /home/claude/ML_HOSPITAL/ml_artifacts/models/xgboost.joblib


## 2. Evaluation Helpers

A single shared function computes all required metrics so results are computed identically across all four models. `CLASS_ORDER = [0, 1, 2]` (LOW -> INTERMEDIARY -> HIGH) is fixed once and reused everywhere so tables and confusion matrices are always in the same order.

In [5]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score,
)

def evaluate_model(name, model, X, y):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X) if hasattr(model, "predict_proba") else None

    pc_f1  = f1_score(y, y_pred, labels=CLASS_ORDER, average=None)
    pc_pre = precision_score(y, y_pred, labels=CLASS_ORDER, average=None, zero_division=0)
    pc_rec = recall_score(y, y_pred, labels=CLASS_ORDER, average=None, zero_division=0)

    row = {
        "Model": name,
        "Accuracy": accuracy_score(y, y_pred),
        "Macro Precision": precision_score(y, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y, y_pred, average="macro"),
        "Weighted F1": f1_score(y, y_pred, average="weighted"),
        "F1 (LOW)": pc_f1[0], "F1 (INTERMEDIARY)": pc_f1[1], "F1 (HIGH)": pc_f1[2],
        "Recall (LOW)": pc_rec[0], "Recall (INTERMEDIARY)": pc_rec[1], "Recall (HIGH)": pc_rec[2],
        "Precision (LOW)": pc_pre[0], "Precision (INTERMEDIARY)": pc_pre[1], "Precision (HIGH)": pc_pre[2],
    }

    if y_proba is not None:
        row["Macro ROC-AUC (OvR)"] = roc_auc_score(
            y, y_proba, multi_class="ovr", average="macro", labels=CLASS_ORDER
        )

    return row, y_pred, y_proba

def plot_confusion_matrix(y_true, y_pred, model_name, filename):
    cm = confusion_matrix(y_true, y_pred, labels=CLASS_ORDER)
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=True,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        linewidths=0.5, ax=ax,
    )
    ax.set_xlabel("Predicted Class")
    ax.set_ylabel("Actual Class")
    ax.set_title(f"Confusion Matrix -- {model_name}")
    fig.tight_layout()
    save_fig(fig, filename)
    return cm

print("Helpers defined.")


Helpers defined.

## 3. Evaluate All Models on the Untouched Test Set

In [6]:
CM_FILENAMES = {
    "Decision Tree":                   "04_decision_tree_confusion_matrix.png",
    "Random Forest":                   "05_random_forest_confusion_matrix.png",
    "Multinomial Logistic Regression": "06_logistic_regression_confusion_matrix.png",
    "XGBoost":                         "07_xgboost_confusion_matrix.png",
}

all_results = []
predictions = {}
probabilities = {}
confusion_matrices = {}

for name, model in models.items():
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")

    row, y_pred, y_proba = evaluate_model(name, model, X_test, y_test)
    all_results.append(row)
    predictions[name] = y_pred
    probabilities[name] = y_proba

    print(classification_report(
        y_test, y_pred, labels=CLASS_ORDER,
        target_names=CLASS_NAMES, zero_division=0,
    ))

    confusion_matrices[name] = plot_confusion_matrix(
        y_test, y_pred, name, CM_FILENAMES[name]
    )

results_df = pd.DataFrame(all_results)



  Decision Tree


              precision    recall  f1-score   support

         LOW       0.35      0.41      0.38        44
INTERMEDIARY       0.50      0.56      0.53       116
        HIGH       0.68      0.58      0.62       146

    accuracy                           0.55       306
   macro avg       0.51      0.51      0.51       306
weighted avg       0.56      0.55      0.55       306



Figure saved -> /home/claude/ML_HOSPITAL/figures/04_decision_tree_confusion_matrix.png

  Random Forest


              precision    recall  f1-score   support

         LOW       0.32      0.36      0.34        44
INTERMEDIARY       0.68      0.66      0.67       116
        HIGH       0.80      0.78      0.79       146

    accuracy                           0.68       306
   macro avg       0.60      0.60      0.60       306
weighted avg       0.68      0.68      0.68       306

Figure saved -> /home/claude/ML_HOSPITAL/figures/05_random_forest_confusion_matrix.png

  Multinomial Logistic Regression


              precision    recall  f1-score   support

         LOW       0.22      0.41      0.29        44
INTERMEDIARY       0.66      0.48      0.56       116
        HIGH       0.76      0.72      0.74       146

    accuracy                           0.58       306
   macro avg       0.54      0.54      0.53       306
weighted avg       0.64      0.58      0.60       306



Figure saved -> /home/claude/ML_HOSPITAL/figures/06_logistic_regression_confusion_matrix.png

  XGBoost
              precision    recall  f1-score   support

         LOW       0.28      0.25      0.27        44
INTERMEDIARY       0.66      0.68      0.67       116
        HIGH       0.78      0.79      0.78       146

    accuracy                           0.67       306
   macro avg       0.57      0.57      0.57       306
weighted avg       0.66      0.67      0.67       306



Figure saved -> /home/claude/ML_HOSPITAL/figures/07_xgboost_confusion_matrix.png


## 4. Model Comparison Table

Sorted by **Macro F1** (primary metric). Accuracy is reported as a secondary metric.

In [7]:
comparison_cols = ["Model", "Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted F1"]
if "Macro ROC-AUC (OvR)" in results_df.columns:
    comparison_cols.append("Macro ROC-AUC (OvR)")

comparison_table = (
    results_df[comparison_cols]
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)

print("Overall model comparison (sorted by Macro F1 -- primary metric):")
display(comparison_table.round(4))


Overall model comparison (sorted by Macro F1 -- primary metric):


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1,Macro ROC-AUC (OvR)
0,Random Forest,0.6765,0.5995,0.6028,0.6006,0.6803,0.7928
1,XGBoost,0.6699,0.5742,0.5729,0.5732,0.6664,0.7880
2,Multinomial Logistic Regression,0.5850,0.5446,0.5370,0.5266,0.6039,0.7158
3,Decision Tree,0.5458,0.5079,0.5149,0.5086,0.5511,0.6595


In [8]:
per_class_f1_cols = ["Model", "F1 (LOW)", "F1 (INTERMEDIARY)", "F1 (HIGH)", "Macro F1"]
per_class_f1_table = (
    results_df[per_class_f1_cols]
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)
print("Per-class F1 comparison:")
display(per_class_f1_table.round(4))


Per-class F1 comparison:


,Model,F1 (LOW),F1 (INTERMEDIARY),F1 (HIGH),Macro F1
0,Random Forest,0.3404,0.6725,0.7889,0.6006
1,XGBoost,0.2651,0.6695,0.7850,0.5732
2,Multinomial Logistic Regression,0.2857,0.5572,0.7368,0.5266
3,Decision Tree,0.3750,0.5285,0.6222,0.5086


In [9]:
all_per_class_cols = [
    "Model",
    "Precision (LOW)", "Recall (LOW)", "F1 (LOW)",
    "Precision (INTERMEDIARY)", "Recall (INTERMEDIARY)", "F1 (INTERMEDIARY)",
    "Precision (HIGH)", "Recall (HIGH)", "F1 (HIGH)",
]
per_class_full = results_df[all_per_class_cols].set_index("Model").round(4)
print("Full per-class Precision / Recall / F1:")
display(per_class_full)


Full per-class Precision / Recall / F1:


,Precision (LOW),Recall (LOW),F1 (LOW),Precision (INTERMEDIARY),Recall (INTERMEDIARY),F1 (INTERMEDIARY),Precision (HIGH),Recall (HIGH),F1 (HIGH)
Model,,,,,,,,,
Decision Tree,0.3462,0.4091,0.3750,0.5000,0.5603,0.5285,0.6774,0.5753,0.6222
Random Forest,0.3200,0.3636,0.3404,0.6814,0.6638,0.6725,0.7972,0.7808,0.7889
Multinomial Logistic Regression,0.2195,0.4091,0.2857,0.6588,0.4828,0.5572,0.7554,0.7192,0.7368
XGBoost,0.2821,0.2500,0.2651,0.6583,0.6810,0.6695,0.7823,0.7877,0.7850


In [10]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_df = comparison_table.sort_values("Macro F1")
bars = ax.barh(plot_df["Model"], plot_df["Macro F1"], color="#34495e")
for bar in bars:
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{bar.get_width():.4f}", va="center", fontsize=10)
ax.set_xlabel("Macro F1 (Test Set)")
ax.set_title("Model Comparison -- Macro F1 on the Untouched Test Set")
ax.set_xlim(0, min(1.0, plot_df["Macro F1"].max() + 0.15))
save_fig(fig, "08_model_macro_f1_comparison.png")


Figure saved -> /home/claude/ML_HOSPITAL/figures/08_model_macro_f1_comparison.png


## 5. ROC-AUC Analysis

One-vs-Rest (OvR) ROC-AUC is reported as a **secondary** diagnostic. **Macro F1 and per-class recall remain the primary comparison criteria.**

In [11]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_test_bin = label_binarize(y_test, classes=CLASS_ORDER)
n_models = len(models)

fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4.5), sharey=True)
if n_models == 1:
    axes = [axes]

roc_rows = []
for ax, (name, proba) in zip(axes, probabilities.items()):
    class_aucs = {}
    if proba is None:
        ax.text(0.5, 0.5, "predict_proba\nnot available",
                ha="center", va="center", transform=ax.transAxes)
    else:
        for i, cls_name in enumerate(CLASS_NAMES):
            fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba[:, i])
            roc_val = auc(fpr, tpr)
            class_aucs[cls_name] = roc_val
            ax.plot(fpr, tpr, label=f"{cls_name} (AUC={roc_val:.3f})")
        ax.plot([0, 1], [0, 1], "--", color="grey", linewidth=1)
        ax.legend(fontsize=8, loc="lower right")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("False Positive Rate")
    roc_rows.append({
        "Model": name,
        **{f"AUC ({k})": v for k, v in class_aucs.items()},
        "Macro AUC": np.mean(list(class_aucs.values())) if class_aucs else np.nan,
    })

axes[0].set_ylabel("True Positive Rate")
fig.suptitle("One-vs-Rest ROC Curves -- All Models", y=1.02)
fig.tight_layout()
save_fig(fig, "09_roc_curves_all_models.png")

roc_df = pd.DataFrame(roc_rows).set_index("Model").round(4)
print("Per-class and Macro AUC (OvR):")
display(roc_df)


Figure saved -> /home/claude/ML_HOSPITAL/figures/09_roc_curves_all_models.png
Per-class and Macro AUC (OvR):


,AUC (LOW),AUC (INTERMEDIARY),AUC (HIGH),Macro AUC
Model,,,,
Decision Tree,0.6353,0.6380,0.7053,0.6595
Random Forest,0.7565,0.7737,0.8482,0.7928
Multinomial Logistic Regression,0.6458,0.7276,0.7740,0.7158
XGBoost,0.7372,0.7768,0.8501,0.7880


## 6. Best-Performing Model

In [12]:
best_row = comparison_table.iloc[0]
second_row = comparison_table.iloc[1]
gap = best_row["Macro F1"] - second_row["Macro F1"]

print(f"Best model by Macro F1 : {best_row['Model']}")
print(f"  Test Macro F1        : {best_row['Macro F1']:.4f}")
print(f"  Test Accuracy        : {best_row['Accuracy']:.4f}")
print(f"  Test Weighted F1     : {best_row['Weighted F1']:.4f}")
if "Macro ROC-AUC (OvR)" in best_row:
    print(f"  Macro ROC-AUC (OvR)  : {best_row['Macro ROC-AUC (OvR)']:.4f}")

print(f"\nMargin over second-best ({second_row['Model']}): {gap:.4f} Macro F1 points")
if gap < 0.02:
    print("\nNote: the margin is small (<0.02). The top two models should be considered "
          "comparably strong; final choice may reasonably depend on secondary criteria "
          "(interpretability, per-class recall balance, training cost).")

print("\nFull ranking:")
display(comparison_table[["Model", "Macro F1", "Accuracy", "Weighted F1"]].round(4))


Best model by Macro F1 : Random Forest
  Test Macro F1        : 0.6006
  Test Accuracy        : 0.6765
  Test Weighted F1     : 0.6803
  Macro ROC-AUC (OvR)  : 0.7928

Margin over second-best (XGBoost): 0.0274 Macro F1 points

Full ranking:


,Model,Macro F1,Accuracy,Weighted F1
0,Random Forest,0.6006,0.6765,0.6803
1,XGBoost,0.5732,0.6699,0.6664
2,Multinomial Logistic Regression,0.5266,0.5850,0.6039
3,Decision Tree,0.5086,0.5458,0.5511


In [13]:
best_name = best_row["Model"]
best_metrics = results_df.set_index("Model").loc[best_name]
print(f"Per-class recall for the best model ({best_name}):")
for cls in CLASS_NAMES:
    print(f"  Recall ({cls:13s}): {best_metrics[f'Recall ({cls})']:.4f}")


Per-class recall for the best model (Random Forest):
  Recall (LOW          ): 0.3636
  Recall (INTERMEDIARY ): 0.6638
  Recall (HIGH         ): 0.7808


## 7. Feature Importance — Tree-Based Models

Built-in `feature_importances_` (total impurity reduction attributable to each feature across all splits/trees). Importance values describe the model's learned predictive reliance on each feature -- **not** a causal clinical relationship.

In [14]:
def get_feature_names(pipeline):
    # Recover output feature names from the pipeline's preprocessing steps.
    try:
        pre_steps = [s for s in pipeline.steps if s[0] not in ("smote", "clf")]
        if not pre_steps:
            return X_test.columns.tolist()
        last_pre = pre_steps[-1][1]
        if hasattr(last_pre, "get_feature_names_out"):
            return list(last_pre.get_feature_names_out())
    except Exception:
        pass
    return X_test.columns.tolist()

FI_FILENAMES = {
    "Decision Tree": "10_decision_tree_feature_importance.png",
    "Random Forest": "11_random_forest_feature_importance.png",
    "XGBoost": "12_xgboost_feature_importance.png",
}

importance_tables = {}
for name in ["Decision Tree", "Random Forest", "XGBoost"]:
    pipeline = models[name]
    clf = pipeline.named_steps["clf"]
    feature_names = get_feature_names(pipeline)
    importances = pd.Series(clf.feature_importances_, index=feature_names).sort_values(ascending=False)
    importance_tables[name] = importances

    fig, ax = plt.subplots(figsize=(7, 5))
    top = importances.head(14)[::-1]
    ax.barh(top.index, top.values, color="#16a085")
    ax.set_xlabel("Feature Importance (Impurity Reduction)")
    ax.set_title(f"Feature Importance -- {name}")
    fig.tight_layout()
    save_fig(fig, FI_FILENAMES[name])

    print(f"\n{name} -- top 5 features:")
    print(importances.head(5).round(4).to_string())


Figure saved -> /home/claude/ML_HOSPITAL/figures/10_decision_tree_feature_importance.png

Decision Tree -- top 5 features:
Total Cholesterol (mg/dL)      0.1878
Fasting Blood Sugar (mg/dL)    0.1441
Systolic BP                    0.1033
Smoking Status                 0.0906
Family History of CVD          0.0783


Figure saved -> /home/claude/ML_HOSPITAL/figures/11_random_forest_feature_importance.png

Random Forest -- top 5 features:
Total Cholesterol (mg/dL)    0.1326
Systolic BP                  0.1146
Age                          0.0952
HDL (mg/dL)                  0.0812
Family History of CVD        0.0768


Figure saved -> /home/claude/ML_HOSPITAL/figures/12_xgboost_feature_importance.png

XGBoost -- top 5 features:
Smoking Status               0.1352
Diabetes Status              0.1348
Family History of CVD        0.1189
Physical Activity Level      0.0718
Total Cholesterol (mg/dL)    0.0631


## 8. Logistic Regression — Coefficients

Each column is one class; each row is one feature. **Interpretive caveat:** continuous variables were standardized before fitting, so their coefficient magnitudes reflect a one-standard-deviation increase. Binary/ordinal variables were **not** standardized. Coefficient magnitudes are therefore **not directly comparable** between continuous and binary/ordinal features. None of this is causal evidence.

In [15]:
lr_pipeline = models["Multinomial Logistic Regression"]
lr_clf = lr_pipeline.named_steps["clf"]
feature_names = get_feature_names(lr_pipeline)

coef_df = pd.DataFrame(
    lr_clf.coef_,
    index=[RISK_LABELS[int(c)] for c in lr_clf.classes_],
    columns=feature_names,
).T.round(4)

print("Logistic Regression -- Multinomial Coefficients (columns = predicted class):")
display(coef_df)


Logistic Regression -- Multinomial Coefficients (columns = predicted class):


,LOW,INTERMEDIARY,HIGH
Age,0.0750,-0.0982,0.0232
Weight (kg),-0.1210,-0.0551,0.1762
Height (cm),-0.0681,0.0707,-0.0025
BMI,0.0166,-0.0357,0.0191
Total Cholesterol (mg/dL),-0.1017,-0.1722,0.2738
HDL (mg/dL),0.2340,0.0801,-0.3141
Fasting Blood Sugar (mg/dL),0.0862,-0.0775,-0.0087
Systolic BP,0.3243,-0.1072,-0.2171
Diastolic BP,0.2194,-0.0887,-0.1307
Sex,0.0305,-0.1181,0.0876


## 9. Save Evaluation Results

In [16]:
comparison_table.round(4).to_csv(ARTIFACTS_DIR / "test_evaluation_summary.csv", index=False)
per_class_full.round(4).to_csv(ARTIFACTS_DIR / "test_per_class_metrics.csv")

print("Saved:")
for name in ["test_evaluation_summary.csv", "test_per_class_metrics.csv"]:
    p = ARTIFACTS_DIR / name
    print(f"  {p}  (exists: {p.exists()})")


Saved:
  /home/claude/ML_HOSPITAL/ml_artifacts/test_evaluation_summary.csv  (exists: True)
  /home/claude/ML_HOSPITAL/ml_artifacts/test_per_class_metrics.csv  (exists: True)


## 10. Limitations

1. **Two-segment file structure** -- the source data appears to concatenate two cohorts (rows 0-985 and 986-1528) with different age/BP/glucose ranges, BMI-recording reliability, and target distributions. All models are trained and evaluated on the pooled mixture; performance may not generalize to a population resembling only one segment.
2. **SMOTENC generates synthetic minority-class training samples** -- interpolated from real training examples. This improves boundary learning for the minority LOW class but adds no genuinely new clinical information.
3. **Weak individual numerical predictors** -- as established in `03_EDA.ipynb`, most continuous variables have only weak univariate correlation with the target (|r| < 0.2). Models rely on combining multiple weak signals.
4. **Single train/test split** -- results are reported on one stratified 80/20 split. A repeated nested cross-validation study would give a more precise estimate of metric variability.
5. **No causal inference** -- all interpretability results (feature importances, LR coefficients) describe learned statistical associations, not causal clinical relationships.

## Summary

| Step | Result |
|---|---|
| Test set | 306 observations -- untouched since Notebook 4 |
| Models evaluated | Decision Tree, Random Forest, Logistic Regression, XGBoost |
| Primary metric | Macro F1 |
| Secondary metrics | Accuracy, Weighted F1, per-class P/R/F1, ROC-AUC (OvR) |
| Figures saved | Confusion matrices (4), Macro F1 chart, ROC curves, Feature importance (3) |
| Best model | Determined programmatically above from actual test-set results |

The printed comparison table above reflects actual execution results and can be cited directly in the thesis Results chapter.